# JEKOVA

In [1]:
# IMPORTS ###########################################

import importlib
import warnings
warnings.filterwarnings("ignore")

#####################################################

import os
import numpy as np
import pandas as pd
from scipy import signal
import matplotlib.pyplot as plt

from pxg import plot
importlib.reload(plot)

from pxg import Stop
from pxg import FS, MS
from pxg import EXG, Rids, Record
from pxg.rec import load_epi

from IPython.display import Markdown, display

%config InlineBackend.figure_format = "retina"

# show dataframes in full, no row/column truncation
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

In [2]:
# Visualization

def PlotPage(
    rec: Record, 
    page = 0, 
    offset = 0,
    marker = False,
    include = []
):
    res = plot.Page(
        rec, 
        page, 
        offset,

        SENSOR = False,
        SIGNAL = True,
        DECTOR = False,
        EXTEND = "JEKOVA",

        # LOW = -700,
        ZEROS = [0],

        # label = "BPM",
        angle = 0,

        rrqs  = False, 
        qsvl  = False, 
        
        punts = False,  
        trig  = False,
        onoff = False,
        letra = False,

        grid  = False,
        anref = False,
        simple = True,

        # simple = False,
        # marker = marker and page not in include,

        show = False,
    )

    if res:
        PON, POF = plot.Range(rec, page, offset)

        # plot.Signal(rec.Local[PON:POF] / MV, zero=-400, color="tab:blue", format="-", linewidth=0.6)
        # plot.Signal(rec.Cover[PON:POF] / MV, zero=-400, color="tab:orange", format="-", linewidth=0.5)

        # plot.Signal(rec.Maxim[PON:POF] / MV, zero=-400, color="tab:red", format="-", linewidth=0.8)
        # plot.Signal(rec.Minim[PON:POF] / MV, zero=-400, color="tab:red", format="-", linewidth=0.8)

        plot.Signal(rec.Digit[PON:POF] * 90, zero=-700, color="tab:gray", format="-", linewidth=0.5, fill=True, alpha=0.2)
        plot.Signal(rec.Class[PON:POF] * 90, zero=-700, color="tab:red", format="-", linewidth=0.5, fill=True, alpha=0.2)
    pass #if

    plot.Show()

    return res
pass #def


def PlotRecord(rec: Record, page: int | list[int] = -1, off = 0, marker = False, title = ""):
    offset = off*plot.FS
    display(Markdown(f"## {rec.DB.upper()} {rec.RID}\n---"))
    if title:
        display(Markdown(f"```text\n{title}\n```"))
    pass #if
    if isinstance(page, list):
        for p in page:
            PlotPage(rec, p, offset, marker = marker)
        pass #for
    elif page != -1:
        PlotPage(rec, page, offset, marker = marker)
    else:
        for page in range(0, (len(rec.Signal) + plot.CHUNK // 2) // plot.CHUNK, 1):
            PlotPage(rec, page, offset, marker = marker)
        pass #for
    pass #if
    return rec
pass #def


In [3]:
# Filters

LYN_WIND = 40 // MS

def LynnFilter(x: np.ndarray, w: int = LYN_WIND) -> np.ndarray:
    w -= w % 2

    a = np.array([1, -2, 1])
    b = np.zeros(w + 1)
    b[0] = 1
    b[w // 2] = -2
    b[w] = 1
    g = (len(b) // 2) ** 2
    shift = w // 2 - 1

    y = signal.lfilter(b, a, x) / g # type: ignore
    y = np.roll(y, -shift)
    return y
pass #def


FS_TARGET = 250    # Hz; every JEKOVA constant is defined for a 250 Hz sampling rate
HP_HZ = 1.0        # first-order high-pass cutoff (per stage)
LP_HZ = 30.0       # Butterworth low-pass cutoff
LP_ORDER = 2       # second-order Butterworth, as the paper specifies
NOTCH_HZ = 50.0    # powerline notch centre frequency (Sofia/European mains)
NOTCH_Q = 30.0     # notch quality factor (dimensionless); bandwidth = NOTCH_HZ / Q,
                   # so Q = 30 gives a ~1.7 Hz notch. The paper does not specify Q.

# Jekova band-pass (paper eq. 1) in integer form, carrying a gain of 16 like the
# Lynn filters. Paper equation, scaled by 16 to clear the fractions:
#     8*FS[i] = 14*FS[i-1] - 7*FS[i-2] + (S[i] - S[i-2])/2
#    16*FS[i] = 28*FS[i-1] - 14*FS[i-2] + (S[i] - S[i-2])
# Coefficients are integers; the peak gain at the 14.6 Hz centre is JEK_GAIN.
# Divide by JEK_GAIN to get unity peak gain. The Step 6 counts are all relative
# to a per-window max / mean / MD, so the cascade is unaffected by the scale.
JEK_B = [16, 0, -16]
JEK_A = [16, -28, 14]
JEK_GAIN = 16 // 4

def JakovaFilter(x, prep = True) -> tuple[np.ndarray, np.ndarray]:
    y = np.asarray(x, dtype=float)

    if prep:
        ## High pass filter ###############
        hp_b, hp_a = signal.butter(1, HP_HZ / (0.5 * FS_TARGET), btype="highpass") # type: ignore
        y = signal.lfilter(hp_b, hp_a, y)
        y = signal.lfilter(hp_b, hp_a, y)

        ## Low pass filter ################
        lp_b, lp_a = signal.butter(LP_ORDER, LP_HZ / (0.5 * FS_TARGET), btype="lowpass") # type: ignore
        y = signal.lfilter(lp_b, lp_a, y)
        ###################################

        ## Notch filter ###################
        nt_b, nt_a = signal.iirnotch(NOTCH_HZ, NOTCH_Q, fs=FS_TARGET) # type: ignore
        y = signal.lfilter(nt_b, nt_a, y)
    else:
        y = LynnFilter(y)
    pass #if

    ## Jekova Equation ################
    k = signal.lfilter(JEK_B, JEK_A, y) / JEK_GAIN # type: ignore
    ###################################

    return np.array(y), np.array(k)
pass #def

In [4]:
def PlotDatabase(db: str, mask: list[str] = []):
    # rids = EXG(db, learn = True, cfm = True, devx = False, wrx = False)
    rids = Rids(db)
    for rid in rids:
        if mask and rid not in mask: continue
        
        rec = Record(db, rid)
        rec.Signal, rec.Jekova = JakovaFilter(rec.Point, False)
        
        rec.DetEpi = load_epi(f"../work/output/pxg/anv/{db}/{rid}.anv", False)
        for page in range(0, (len(rec.Signal) + plot.CHUNK // 2) // plot.CHUNK, 1):
            PON, POF = plot.Range(rec, page, 0)
            cc = 0
            for epi in rec.RefEpi:
                if epi.End < PON or epi.Time >= POF: continue
                if epi.Name in ["#VT", "VFL", "VF", "WF"]: cc+=1
            pass #for
            for epi in rec.DetEpi:
                if epi.End < PON or epi.Time >= POF: continue
                if epi.Name in ["#VT", "VFL", "VF", "WF"]: cc+=1
            pass #for
            if cc == 0: continue
            PlotRecord(rec, page = [page], off = 0, marker = False, title = "")
        pass #for
    pass #for
pass #def

In [5]:
# Jekova Algorithm

COLUMNS = [
    "db", 
    "rid", 
    "pon", 
    "pof", 
    "vfb", 

    # "smin",
    "smax",
    "savg",
    "sdev",

    "fmax",
    "favg",
    "mdev",

    "cnt1",
    "cnt2", 
    "cnt3",
    "cnrt",
]

# calculate Jekova counts in a 1 s segment of the band-pass output (paper step 5-6)
def Counts(db, rid, pon, pof, vfb, lyn: np.ndarray, seg: np.ndarray) -> dict:
    # step 5: the counts are defined on the absolute (rectified) filter output
    abs_fs = np.abs(np.asarray(seg, dtype=float))
    abs_ly = np.abs(np.asarray(lyn, dtype=float))

    # smin = np.min(lyn)
    smax = np.max(abs_ly)
    savg = np.mean(abs_ly)
    sdev = np.mean(np.abs(abs_ly - savg))   # mean absolute deviation, not median

    fmax = np.max(abs_fs)
    favg = np.mean(abs_fs)
    mdev = np.mean(np.abs(abs_fs - favg))   # mean absolute deviation, not median

    # step 6: number of samples falling in each amplitude band
    cnt1 = int(np.sum(abs_fs >= fmax / 2))                                  # 0.5*smax .. smax
    cnt2 = int(np.sum(abs_fs >= favg))                                      # smean    .. smax
    cnt3 = int(np.sum((abs_fs >= favg - mdev) & (abs_fs <= favg + mdev)))   # smean +- MD

    return dict(
        db=db, 
        rid=rid, 
        pon=pon, 
        pof=pof, 
        vfb=vfb, 

        # smin=int(smin), 
        smax=int(smax), 
        savg=int(savg),
        sdev=int(sdev),

        fmax=int(fmax),
        favg=int(favg), 
        mdev=int(mdev),

        cnt1=cnt1, 
        cnt2=cnt2, 
        cnt3=cnt3,

        cnrt = cnt1 * cnt2 / cnt3 if cnt3 > 0 else 0
    )
pass #def

def mean(x: list[float]) -> float:
    return sum(x)
pass #def

AGGREGATE: dict = dict(
    vfb = sum,  # samples of reference VF inside the window

    # smin = min,
    smax = max,  
    savg = mean,
    sdev = mean,

    fmax = max,
    favg = mean,
    mdev = mean,

    cnt1  = sum,  # Jekova step 6 counts
    cnt2  = sum,
    cnt3  = sum,
)

def Window(rows: list[dict], s: int, n: int) -> dict:
    """Aggregate the n consecutive 1 s rows starting at s into one window row."""
    w = rows[s:s + n]
    out = dict(db=w[0]["db"], rid=w[0]["rid"], pon=w[0]["pon"], pof=w[-1]["pof"])
    for name, agg in AGGREGATE.items():
        out[name] = agg(r[name] for r in w)
        if agg == mean:
            out[name] = int(out[name] / n)
        pass #if
    pass #for
    out["cnrt"] = out["cnt1"] * out["cnt2"] / out["cnt3"] if out["cnt3"] > 0 else 0
    return out
pass #def

def Process(rec: Record) -> tuple[list[dict], list[dict], list[dict], np.ndarray]:
    lyn, jek = JakovaFilter(rec.Point, False)
    rec.Jekova = jek

    vfib = np.zeros(len(rec.Point), dtype=int)
    for epi in rec.RefEpi:
        if epi.Name in ["#VT", "VFL", "VF", "WF"]:
            vfib[epi.Time:epi.End+1] = 1
        pass #if
    pass #for
    rec.RefVFB = vfib

    rows = []
    for s in range(len(rec.Point) // FS):
        pon = s * FS
        pof = pon + FS
        vfb = int(np.sum(vfib[pon:pof]))
        row = Counts(rec.DB, rec.RID, pon, pof, vfb, lyn[pon:pof], jek[pon:pof])
        rows.append(row)
    pass #for

    row5 = [Window(rows, s, 5) for s in range(len(rows) - 5)]
    row8 = [Window(rows, s, 8) for s in range(len(rows) - 8)]

    return rows, row5, row8, rec.RefVFB
pass #def

def Prepare(db: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    rows = []
    row5 = []
    row8 = []

    # mkdir ../work/vfib/{db}
    os.makedirs(f"../work/vfib/{db}", exist_ok=True)

    rids = Rids(db)
    for rid in rids:
        rec = Record(db, rid)
        rs, r5, r8, vfb = Process(rec)
        rows.extend(rs)
        row5.extend(r5)
        row8.extend(r8)

        # save the per-record VFB mask as a compact binary (uint8)
        np.save(f"../work/vfib/{db}/{rid}.ref.npy", vfb.astype(np.uint8))
    pass #for
    dfs = pd.DataFrame(rows, columns=COLUMNS)
    df5 = pd.DataFrame(row5, columns=COLUMNS)
    df8 = pd.DataFrame(row8, columns=COLUMNS)

    # Save to work/vfib/{db}
    dfs.to_csv(f"../work/vfib/{db}_dfs.tsv", index=False, sep="\t", float_format="%.5f")
    df5.to_csv(f"../work/vfib/{db}_df5.tsv", index=False, sep="\t", float_format="%.5f")
    df8.to_csv(f"../work/vfib/{db}_df8.tsv", index=False, sep="\t", float_format="%.5f")

    return dfs, df5, df8
pass #def

In [6]:
DATABASES = ["mitdb", "ahadb", "cudb"] # "edb", "vfdb"]

for db in DATABASES:
    print(db)
    Prepare(db)
pass #for

mitdb
ahadb
cudb


In [7]:
# Evaluate the published Jekova cascade, thresholds scaled to the window length.

VER = 8

EPOCH = 10 * FS          # the paper's 10 s epoch at 250 Hz = 2500 samples
SPAN = VER * FS            # the 8 s window we use for the cascade, 2000 samples
FRAC = 0.85

# Published thresholds (paper section 2.2.5), valid for a 2500-sample epoch. Every
# count is a number of samples, so all of them scale linearly with the window
# length; the ratio Count1*Count2/Count3 scales linearly too, since N*N/N = N.
JEKOVA_THRESHOLDS = dict(
    C1_LO = 250 * SPAN / EPOCH, 
    C1_HI = 400 * SPAN / EPOCH, 
    C2_LO = 600 * SPAN / EPOCH, 
    C2_HI = 950 * SPAN / EPOCH, 
    C2_MAX = 1100 * SPAN / EPOCH,
    RATIO = 210,
)

def Classify(wf: pd.DataFrame, thr: dict = JEKOVA_THRESHOLDS) -> np.ndarray:
    smax = wf["smax"].to_numpy(dtype=float)

    cnt1 = wf["cnt1"].to_numpy(dtype=float)
    cnt2 = wf["cnt2"].to_numpy(dtype=float)
    cnt3 = wf["cnt3"].to_numpy(dtype=float)
    cnrt = wf["cnrt"].to_numpy(dtype=float)

    r0 = (smax < 0)
    r1 = (cnt1 <  thr["C1_LO"]) & (cnt2 > thr["C2_HI"]) & (cnrt < thr["RATIO"])
    r2 = (cnt1 >= thr["C1_LO"]) & (cnt1 < thr["C1_HI"]) & (cnt2 < thr["C2_LO"]) & (cnrt < thr["RATIO"])
    r3 = (cnt1 >= thr["C1_LO"]) & (cnt2 > thr["C2_HI"])
    r4 = (cnt2 >= thr["C2_MAX"])

    call = np.full(len(wf), "?", dtype="<U1")
    for rule, label in [
            (r0, "N"),
            (r1, "N"), 
            (r2, "N"), 
            (r3, "S"), 
            (r4, "S"),
        ]:   # first match wins
        call = np.where((call == "?") & rule, label, call)
    pass #for

    return call
pass #def

In [8]:
# Evaluate 

def Rates(tp: int, fn: int, fp: int, tn: int) -> dict:
    """Confusion counts and derived rates, shared by the window and sample tables."""
    return dict(
        TN=tn, TP=tp, FN=fn, FP=fp, ERR = fp + fn,
        SPC = round(100 * tn / (tn + fp), 2) if tn + fp else float("nan"),
        SEN = round(100 * tp / (tp + fn), 2) if tp + fn else float("nan"),
        PPV = round(100 * tp / (tp + fp), 2) if tp + fp else float("nan"),
        F1  = round(100 * 2 * tp / (2 * tp + fp + fn), 2) if 2 * tp + fp + fn else float("nan"),
    )
pass #def

def MetricsWin(call, shock, nonsh, mixed) -> dict:
    """Window-level confusion for one boolean slice of a database."""
    tp = int(((call == "S") & shock).sum())
    fn = int(((call != "S") & shock).sum())
    fp = int(((call == "S") & nonsh).sum())
    tn = int(((call != "S") & nonsh).sum())
    return dict(nonsh=int((nonsh & ~mixed).sum()), mixed=int(mixed.sum()), shock=int(shock.sum()), **Rates(tp, fn, fp, tn))
pass #def

def EvaluateWin(df: pd.DataFrame) -> pd.DataFrame:
    """Per-record window rows (one per rid) plus a TOTAL row."""
    call = Classify(df)

    # frac is a cut, not a band: every window lands in exactly one class.
    shock = (df["vfb"] >= FRAC * SPAN).to_numpy()
    nonsh = ~shock
    # windows straddling an episode edge: some VF present but under the cut,
    # scored as negatives, reported apart from the clear negatives
    mixed = (df["vfb"].to_numpy() > 0) & nonsh

    db  = df["db"].iloc[0]
    rid = df["rid"].to_numpy()

    wins = []
    for r in pd.unique(rid):
        m = rid == r
        wins.append({"db": db, "rid": str(r), **MetricsWin(call[m], shock[m], nonsh[m], mixed[m])})
    pass #for
    wins.append({"db": db, "rid": "TOTAL", **MetricsWin(call, shock, nonsh, mixed)})
    return pd.DataFrame(wins)
pass #def

def Evaluate(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Per-record SAMPLE rows (same columns as EvaluateWin) plus a TOTAL row.

    The window decisions are mapped back to a per-sample prediction mask (a sample
    is shockable if any covering window was called "S"), compared against the
    per-sample reference, and the mask saved as {db}/{rid}.vfb.npy.
    """

    call = Classify(df)

    # frac is a cut, not a band: every window lands in exactly one class.
    shock = (df["vfb"] >= FRAC * SPAN).to_numpy()
    nonsh = ~shock
    # windows straddling an episode edge: some VF present but under the cut,
    # scored as negatives, reported apart from the clear negatives
    mixed = (df["vfb"].to_numpy() > 0) & nonsh

    db  = df["db"].iloc[0]
    rid = df["rid"].to_numpy()
    pon = df["pon"].to_numpy()
    pof = df["pof"].to_numpy()

    wins = []
    samp = []
    stot = [0, 0, 0, 0]                    # pooled sample TP, FN, FP, TN for TOTAL
    for r in pd.unique(rid):
        m = rid == r
        wins.append({"db": db, "rid": str(r), **MetricsWin(call[m], shock[m], nonsh[m], mixed[m])})

        ref  = np.load(f"../work/vfib/{db}/{r}.ref.npy").astype(bool)
        pred = np.zeros(len(ref), dtype=bool)
        for a, b, cl in zip(pon[m], pof[m], call[m]):
            if cl == "S":
                pred[a:b] = True
            pass #if
        pass #for
        np.save(f"../work/vfib/{db}/{r}.vfb.npy", pred.astype(np.uint8))

        # Save as detected annotations
        os.makedirs(f"../work/output/pxg/anv/{db}", exist_ok=True)
        with open(f"../work/output/pxg/anv/{db}/{r}.anv", "w") as f:
            prev = -1
            for ix in range(len(pred)):
                p = int(pred[ix])
                if p == prev: continue
                if p == 0:
                    f.write(f"{ix},+,(NF\n")
                else:
                    f.write(f"{ix},+,(WF\n")
                pass #if
                prev = p
            pass #for
            f.write(f"{len(pred)},+,(END\n")
        pass #with

        tp = int((pred & ref).sum());  fn = int((~pred & ref).sum())
        fp = int((pred & ~ref).sum()); tn = int((~pred & ~ref).sum())
        for i, v in enumerate((tp, fn, fp, tn)):
            stot[i] += v
        pass #for
        samp.append({"db": db, "rid": str(r), "nonsh": tn + fp, "mixed": 0, "shock": tp + fn, **Rates(tp, fn, fp, tn)})
    pass #for

    wins.sort(key=lambda x: x["ERR"], reverse=True)   # sort by error count, descending

    wins.append({"db": db, "rid": "TOTAL", **MetricsWin(call, shock, nonsh, mixed)})
    tp, fn, fp, tn = stot
    samp.append({"db": db, "rid": "TOTAL", "nonsh": tn + fp, "mixed": 0, "shock": tp + fn, **Rates(tp, fn, fp, tn)})

    return pd.DataFrame(wins), pd.DataFrame(samp)
pass #def

In [9]:
wins, samp = [], []
for db in DATABASES:
    df8 = pd.read_csv(f"../work/vfib/{db}_df{VER}.tsv", sep="\t")
    # rw = EvaluateWin(df8)
    rw, rs = Evaluate(df8)
    if db in ["cudb"]:
        display(Markdown(f"### {db} — per window"));   display(rw)
        # display(Markdown(f"### {db} — per sample"));   display(rs)
        pass
    pass #if
    wins.append(rw[rw["rid"] == "TOTAL"])
    samp.append(rs[rs["rid"] == "TOTAL"])
pass #for

# display(Markdown("## Summary — per sample"))
# display(pd.concat(samp, ignore_index=True))

# summary: the TOTAL row of every database, window vs sample
display(Markdown("## Summary — per window"))
display(pd.concat(wins, ignore_index=True))

### cudb — per window

,db,rid,nonsh,mixed,shock,TN,TP,FN,FP,ERR,SPC,SEN,PPV,F1
0,cudb,cu20,237,7,255,242,32,223,2,225,99.18,12.55,94.12,22.15
1,cudb,cu30,112,35,352,139,211,141,8,149,94.56,59.94,96.35,73.91
2,cudb,cu28,489,7,3,354,3,0,142,142,71.37,100.00,2.07,4.05
3,cudb,cu12,297,14,188,309,91,97,2,99,99.36,48.40,97.85,64.77
4,cudb,cu27,468,13,18,408,17,1,73,74,84.82,94.44,18.89,31.48
5,cudb,cu10,309,7,183,304,130,53,12,65,96.20,71.04,91.55,80.00
6,cudb,cu13,437,14,48,406,33,15,45,60,90.02,68.75,42.31,52.38
7,cudb,cu31,487,7,5,438,5,0,56,56,88.66,100.00,8.20,15.15
8,cudb,cu19,407,21,71,409,47,24,19,43,95.56,66.20,71.21,68.61
9,cudb,cu16,371,28,100,383,74,26,16,42,95.99,74.00,82.22,77.89


## Summary — per window

,db,rid,nonsh,mixed,shock,TN,TP,FN,FP,ERR,SPC,SEN,PPV,F1
0,mitdb,TOTAL,85932,69,111,85998,101,10,3,13,100.00,90.99,97.12,93.95
1,ahadb,TOTAL,134939,98,5346,134916,5086,260,121,381,99.91,95.14,97.68,96.39
2,cudb,TOTAL,13432,540,3493,13395,2675,818,577,1395,95.87,76.58,82.26,79.32


```
#	db	rid	nonsh	mixed	shock	TP	FN	FP	TN	ERR	SPC	SEN	PPV	F1
0	mitdb	TOTAL	85932	69	111	85998	101	10	3	13	100.00	90.99	97.12	93.95
1	ahadb	TOTAL	134939	98	5346	134916	5086	260	121	381	99.91	95.14	97.68	96.39
2	cudb	TOTAL	13432	540	3493	13395	2675	818	577	1395	95.87	76.58	82.26	79.32
```

In [77]:
# PlotDatabase("cudb", ["cu28", "#cu20", "#cu30", "#cu27"])

In [10]:
# Stop()

# Decision tree on the counts c1, c2, c3
#
# A shallow tree just learns count thresholds (what the cascade does by hand), and
# its rules are readable. Evaluated two ways: in-sample (comparable to the manual
# thresholds, which were also tuned in-sample) and grouped cross-validation with
# GroupKFold by record, so windows from one record never leak between folds. The
# gap between the two is the per-record drift.

from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import GroupKFold, cross_val_predict

TREE_DBS = ["mitdb", "ahadb", "cudb"]   # edb has no shockable windows

frames = []
for db in TREE_DBS:
    df8 = pd.read_csv(f"../work/vfib/{db}_df8.tsv", sep="\t")
    frames.append(df8)
pass #for

DATA = pd.concat(frames, ignore_index=True)

####

FTRS = [
    "smax", "savg", "sdev",
    # "fmax", "favg", "mdev",
    "cnt1", "cnt2", "cnt3", "cnrt"
]

X = DATA[FTRS].to_numpy(float)
y = (DATA["vfb"].to_numpy() >= FRAC * SPAN).astype(int)
groups = (DATA["db"] + "/" + DATA["rid"].astype(str)).to_numpy()   # one group per record

def tree_report(y_true, y_pred, mask, name) -> dict:
    yy, pp = y_true[mask], y_pred[mask]
    tp = int(((pp == 1) & (yy == 1)).sum()); fp = int(((pp == 1) & (yy == 0)).sum())
    fn = int(((pp == 0) & (yy == 1)).sum()); tn = int(((pp == 0) & (yy == 0)).sum())
    return dict(set=name, shock=int(yy.sum()), TP=tp, FN=fn, FP=fp, TN=tn,
                SPC=round(100 * tn / (tn + fp), 2) if tn + fp else float("nan"),
                SEN=round(100 * tp / (tp + fn), 2) if tp + fn else float("nan"),
                PPV=round(100 * tp / (tp + fp), 2) if tp + fp else float("nan"),
                F1 =round(100 * 2 * tp / (2 * tp + fp + fn), 2) if 2 * tp + fp + fn else float("nan"))
pass #def

clf = DecisionTreeClassifier(max_depth=4, random_state=0)
insample = clf.fit(X, y).predict(X)
crossval = cross_val_predict(clf, X, y, groups=groups, cv=GroupKFold(5))

for tag, pred in [("in-sample", insample), ("grouped-CV", crossval)]:
    rows = [tree_report(y, pred, (DATA["db"] == db).to_numpy(), db) for db in TREE_DBS]
    rows.append(tree_report(y, pred, np.ones(len(y), bool), "POOLED"))
    display(Markdown(f"### Decision tree (depth 4) - {tag}"))
    display(pd.DataFrame(rows))
pass #for

# the learned thresholds
print(export_text(clf.fit(X, y), feature_names=FTRS, max_depth=3))

### Decision tree (depth 4) - in-sample

,set,shock,TP,FN,FP,TN,SPC,SEN,PPV,F1
0,mitdb,111,95,16,10,85991,99.99,85.59,90.48,87.96
1,ahadb,5346,5177,169,121,134916,99.91,96.84,97.72,97.28
2,cudb,3493,2924,569,681,13291,95.13,83.71,81.11,82.39
3,POOLED,8950,8196,754,812,234198,99.65,91.58,90.99,91.28


### Decision tree (depth 4) - grouped-CV

,set,shock,TP,FN,FP,TN,SPC,SEN,PPV,F1
0,mitdb,111,97,14,9,85992,99.99,87.39,91.51,89.40
1,ahadb,5346,4766,580,198,134839,99.85,89.15,96.01,92.45
2,cudb,3493,2896,597,937,13035,93.29,82.91,75.55,79.06
3,POOLED,8950,7759,1191,1144,233866,99.51,86.69,87.15,86.92


|--- cnt2 <= 754.50
|   |--- savg <= 126.50
|   |   |--- cnt2 <= 736.50
|   |   |   |--- smax <= 1149.00
|   |   |   |   |--- class: 0
|   |   |   |--- smax >  1149.00
|   |   |   |   |--- class: 0
|   |   |--- cnt2 >  736.50
|   |   |   |--- savg <= 82.50
|   |   |   |   |--- class: 0
|   |   |   |--- savg >  82.50
|   |   |   |   |--- class: 1
|   |--- savg >  126.50
|   |   |--- cnt2 <= 714.50
|   |   |   |--- smax <= 982.50
|   |   |   |   |--- class: 0
|   |   |   |--- smax >  982.50
|   |   |   |   |--- class: 0
|   |   |--- cnt2 >  714.50
|   |   |   |--- cnt3 <= 1185.00
|   |   |   |   |--- class: 1
|   |   |   |--- cnt3 >  1185.00
|   |   |   |   |--- class: 1
|--- cnt2 >  754.50
|   |--- cnt3 <= 1170.50
|   |   |--- smax <= 84.50
|   |   |   |--- savg <= 4.50
|   |   |   |   |--- class: 0
|   |   |   |--- savg >  4.50
|   |   |   |   |--- class: 1
|   |   |--- smax >  84.50
|   |   |   |--- smax <= 534.50
|   |   |   |   |--- class: 1
|   |   |   |--- smax >  534.50
|   |   |